In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2 * x + 1

class LinearDataset(Dataset):
    def __init__(self, x, y):
        if len(x) != len(y):
            raise ValueError('x and y must have the same length')
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

dataset = LinearDataset(x, y)

print('Dataset size:', len(dataset))
print('First sample:', dataset[0])




Dataset size: 100
First sample: (tensor([-1.]), tensor([-1.]))


In [ ]:
batch_size = 16

# 100个样本16个一批
train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False
)

print('Number of batches:', len(train_loader))

batch_x, batch_y = next(iter(train_loader))
print('Batch x shape:', batch_x.shape)
print('Batch y shape:', batch_y.shape)

batch_sizes = [len(batch_x) for batch_x, _ in train_loader]
print(batch_sizes)

Number of batches: 7
Batch x shape: torch.Size([16, 1])
Batch y shape: torch.Size([16, 1])
[16, 16, 16, 16, 16, 16, 4]


In [4]:
def train(model, dataloader, loss_fn, optimizer, epochs=100):
    history = []

    for epoch in range(epochs):
        total_loss = 0.0

        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()

            y_pred = model(batch_x)
            loss = loss_fn(y_pred, batch_y)

            loss.backward()
            optimizer.step()

            # loss 是当前 batch 的平均损失，乘以样本数后再累计
            total_loss += loss.item() * batch_x.size(0)

        # 按整个 Dataset 的样本数计算 epoch 平均损失
        epoch_loss = total_loss / len(dataloader.dataset)
        history.append(epoch_loss)

        if (epoch + 1) % 20 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.6f}')

    return history

In [5]:
torch.manual_seed(42)

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

with torch.no_grad():
    initial_loss = loss_fn(model(x), y).item()

history = train(
    model,
    train_loader,
    loss_fn,
    optimizer,
    epochs=100
)

with torch.no_grad():
    final_loss = loss_fn(model(x), y).item()

print('Initial loss:', initial_loss)
print('Final loss:', final_loss)
print('Weight:', model.weight.item())
print('Bias:', model.bias.item())

assert final_loss < initial_loss
assert abs(model.weight.item() - 2.0) < 0.1
assert abs(model.bias.item() - 1.0) < 0.1

Epoch [20/100], Loss: 0.000037
Epoch [40/100], Loss: 0.000000
Epoch [60/100], Loss: 0.000000
Epoch [80/100], Loss: 0.000000
Epoch [100/100], Loss: 0.000000
Initial loss: 0.5479642152786255
Final loss: 2.7274181719529056e-13
Weight: 1.9999991655349731
Bias: 0.9999998211860657


In [6]:
for size in (8, 16, 32, 100):
    loader = DataLoader(
        dataset,
        batch_size=size,
        shuffle=False
    )
    sizes = [len(batch_x) for batch_x, _ in loader]
    print(
        f'batch_size={size}, '
        f'batches={len(loader)}, '
        f'last_batch={sizes[-1]}'
    )

batch_size=8, batches=13, last_batch=4
batch_size=16, batches=7, last_batch=4
batch_size=32, batches=4, last_batch=4
batch_size=100, batches=1, last_batch=100
